# Restore CSA architecture on CLIP patches — T4×2 TRAIN smoke

InternetON, Accelerator **GPU T4×2**, Save & Run All. Attach FOUR existing inputs:
- `ducanh2403/visual-cache`
- `vuthetam/mscoco-2014`
- `ducanh2403/csa-semantic-banks-v130`
- `ducanh2403/csa-prompt-full-karpathy-v175`

No API Secret, checkpoint or new CLIP/UPT extraction. Fresh initialization. Budget reported6h yours/30h Ngân; this notebook is bounded to75min including setup, not a full10epoch run or billing estimate.

Reuse our paper-core:2complete object alignment/fusion layers →1object-guided relation layer →TWO separate memories →3layer parallel adaptive decoder with independent per-word gates. Dim512/heads8/FFN2048/dropout.2. Remove old single-memory mixture and strength gate. The two memories are packaged for beam expansion and split BEFORE the two decoder cross-attentions.

Disclosed adaptation: frozenCLIP-B16 patches and CLIP-text banks replace the paper's detector/RegionCLIP. Our image-specific UPT prompt adapters remain AFTER the core encoder and modify the two streams separately. Thus this is CSA structural inheritance + topic-specific prompt extension, not exact full-system replication. No paper-score claim, no new loss.

Only256fixedTRAINimages/200XEupdates/global32/FP32/Adam. Check two-stage semantic carry, independent stream response, causal mask, actual DDP gradient parity, prompt-mask locality, TRAIN-probe loss decrease, beam5 score replay, throughput and GPU memory. No VALID/TEST/CIDEr/SCST/reranking. No full33GiB copy; selectedTRAINpatches only. Source/cache SHA verification remains.

Send `clip_patch_csa_structure_smoke_v1_review.zip`. This smoke does NOT prove better CIDEr and is not a full-training checkpoint. Full-run scheduling follows measured results; no automatic restart, full run or Torch reinstall. Author checks are NumPy/mock/AST only; actual neural checks run here on Kaggle.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
REVISION='0323747b13122800aa6b5e6ffd9b2d2913054ee8'
MAPPING={'experiments/clip_patch_baseline/csa_patch_model.py': 'csa_patch_model.py', 'experiments/clip_patch_baseline/csa_structure_data.py': 'csa_structure_data.py', 'experiments/clip_patch_baseline/csa_structure_runtime.py': 'csa_structure_runtime.py', 'experiments/clip_patch_baseline/run_csa_structure_smoke.py': 'run_csa_structure_smoke.py', 'experiments/clip_patch_baseline/test_csa_structure_data.py': 'test_csa_structure_data.py', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/memory_fusion.py': 'memory_fusion.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/test_semantic_data.py': 'test_semantic_data.py', 'experiments/clip_patch_baseline/CSA_STRUCTURE_PROTOCOL.md': 'CSA_STRUCTURE_PROTOCOL.md', 'experiments/clip_patch_baseline/make_csa_structure_notebook.py': 'make_csa_structure_notebook.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': 'SAVE_BANK_V130_PACKAGE.json', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': 'FULL_PROMPT_FRESH_V175_RESULT.json', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'bank_words_corrected.json'}
HASHES={'experiments/clip_patch_baseline/csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'experiments/clip_patch_baseline/csa_structure_data.py': '3bce011b00f67d272444cdd4d6e17acbbb744c8d9614549fb06c73be79aba1a2', 'experiments/clip_patch_baseline/csa_structure_runtime.py': '85a9806799ad2d2cb5da0bb458c373af830881f5cd17770a597d4b0274309cd9', 'experiments/clip_patch_baseline/run_csa_structure_smoke.py': 'e4c5c9f7fa9ed1fb244b9745170d02f680df36a7305116c645fc66cc5530ab2c', 'experiments/clip_patch_baseline/test_csa_structure_data.py': '55b45cf9cb6570a8f3d8cff1bbca7b80a0a29fd8dd3d8a24bbf942abc5f7cf16', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'experiments/clip_patch_baseline/memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/test_semantic_data.py': 'e9d9132db2e83b47e7f29cd1c465389cb4f16db2cd62a1cf27f048a06e29a572', 'experiments/clip_patch_baseline/CSA_STRUCTURE_PROTOCOL.md': '0519a80e74f7a3c4ed6a39978fe7cc6b864fec142fcb977520def08791328177', 'experiments/clip_patch_baseline/make_csa_structure_notebook.py': '68526f40aca7f4ba17691227db0ebec6ac0d9c12efe9c810cd820f245f44e777', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': '452e94530b07b117be6873f004404b558e29e87cb09e97e27a1548d0bfd5f81a', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac'}
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
BANK_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
PROMPT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
OUTPUT=Path('/kaggle/working/clip_patch_csa_structure_smoke_v1')
for p in (VISUAL_CACHE,COCO_JSON,BANK_ROOT,PROMPT_ROOT):
    if not p.exists():raise FileNotFoundError('Attach/correct all four inputs BEFORE GPU: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve prior output; use fresh session/output path')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<2*1024**3:raise RuntimeError('Need2GiB for small TRAIN stage/logs')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='csa-structure-',dir='/tmp'));LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+4500 #75min STOPcap including setup; not ETA/billing.
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json'
cfg=dict(output=str(OUTPUT),stage=str(SOURCE/'stage'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),
         bank_root=str(BANK_ROOT),prompt_root=str(PROMPT_ROOT),updates=200,prepare_seconds=2100,worker_seconds=2100)
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))
write_config()


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Parent cap exceeded')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('CSA structure active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    for name,destination in MAPPING.items():
        remaining=PARENT_DEADLINE-time.monotonic()
        if remaining<=0:raise TimeoutError('Source download parent cap')
        raw=urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+name,timeout=min(90,remaining)).read()
        if hashlib.sha256(raw).hexdigest()!=HASHES[name]:raise ValueError('Changed source: '+name)
        (SOURCE/destination).write_bytes(raw)
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2))
    run([sys.executable,'-u',SOURCE/'test_csa_structure_data.py'],120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],60)
    run([sys.executable,'-u',SOURCE/'run_csa_structure_smoke.py','prepare','--config',CONFIG],2100)
    cfg['worker_seconds']=min(2100,PARENT_DEADLINE-time.monotonic()-120)
    if cfg['worker_seconds']<300:raise TimeoutError('Insufficient remaining parent budget; preserve log/stage')
    write_config()
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2',
         '--max_restarts=0',SOURCE/'run_csa_structure_smoke.py','worker','--config',CONFIG],cfg['worker_seconds']+30)
    complete=json.loads((OUTPUT/'result.json').read_text())['status']=='CLIP_PATCH_CSA_STRUCTURE_SMOKE_COMPLETE'
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc());raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,full_training_complete=False,test_used=False,validation_used=False,SCST=False,reranking=False)))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file() and p.suffix in ('.json','.jsonl','.txt','.log'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True)
    print('SMOKE COMPLETE; no full training/CIDEr result.' if complete else 'INCOMPLETE; preserve log/output, no automatic retry.',flush=True)
